# NumPy: arreglos y operaciones vectorizadas

Representaremos rondas de temperatura de tres salas. Primero aprenderemos a leer
la forma de un arreglo; después aplicaremos operaciones a todas las mediciones y
reutilizaremos los resultados en una aplicación.

## Contenido

1. Listas y arreglos; dimensiones y tipos.
2. Selección y máscaras.
3. Vistas y copias.
4. Operaciones vectorizadas, resúmenes por eje y broadcasting.
5. Mediciones incompletas y paso a una aplicación.

Abre esta notebook en VS Code desde `01-exploracion` y selecciona el kernel de `.venv`.
Los ejercicios tienen celdas independientes: puedes recorrer los ejemplos sin
completarlos y resolver cada ejercicio en su propia celda.


In [1]:
import sys
from pathlib import Path

import numpy as np

print(f"Python: {sys.executable}")
print(f"Working directory: {Path.cwd()}")
print(f"NumPy: {np.__version__}")

data_dir = Path.cwd().parent / "datos"
if not (data_dir / "readings.csv").is_file():
    raise FileNotFoundError(
        "Start the notebook from the 01-exploracion project directory"
    )

Python: /Users/master/Documents/MIA-curso/MIA-UADY/Programacion_para_IA/unidad_02/sesion_01/01-exploracion/.venv/bin/python
Working directory: /Users/master/Documents/MIA-curso/MIA-UADY/Programacion_para_IA/unidad_02/sesion_01/01-exploracion
NumPy: 2.5.3


## 1. Listas y arreglos

NumPy ofrece `ndarray`, un arreglo de elementos con un tipo común. Su utilidad
no se limita a escribir menos ciclos: también hace explícita la organización de
los datos. Una lista sigue siendo útil cuando necesitamos una colección flexible.

Predice qué devuelve cada multiplicación: una opera sobre una lista y la otra
sobre sus valores numéricos.

[Introducción oficial a NumPy](https://numpy.org/doc/stable/user/absolute_beginners.html).

# Predicción línea 1: ...44
# Predicción línea 2: ...48

In [2]:
temperature_list = [22.0, 24.0, 20.0]
temperature_array = np.array(temperature_list)
print(temperature_list * 2)
print(temperature_array * 2)

[22.0, 24.0, 20.0, 22.0, 24.0, 20.0]
[44. 48. 40.]


#CONCLUSIÓN:

**Predicción vs. resultado**

| Línea | Predicción | Resultado real | ¿Coincide? |
|---|---|---|---|
| `temperature_list * 2` | Se esperaba que multiplicara cada número: 44, 48, 40 | `[22.0, 24.0, 20.0, 22.0, 24.0, 20.0]` | No |
| `temperature_array * 2` | 44, 48, 40 | `[44. 48. 40.]` | Sí |

**Explicación**

- Con una **lista** de Python, `* 2` no realiza una operación matemática: **repite la lista completa** dos veces, del mismo modo que `"ab" * 2` produce `"abab"`.
- Con un **arreglo de NumPy**, `* 2` **multiplica cada número** por 2 sin necesidad de un ciclo `for`. Este comportamiento se conoce como operación vectorizada.
- La predicción falló porque se supuso que la lista se comportaría como un conjunto de números. Una lista es un contenedor general; un arreglo de NumPy está diseñado para realizar cálculos sobre sus elementos.

**Cómo distinguirlos en la salida**

- La lista se imprime **con comas** entre los valores.
- El arreglo se imprime **sin comas**, y el punto en `44.` indica que los valores son decimales.

## 2. Dimensiones, forma y tamaño

Usaremos este orden de columnas: sala norte, sala sur y almacén. Cada fila
corresponde a una ronda en la que se midieron las tres salas.

| Ronda | `north_room` | `south_room` | `storage_room` |
|---|---:|---:|---:|
| 0 | 22 | 24 | 20 |
| 1 | 23 | 25 | 21 |
| 2 | 24 | 26 | 22 |

`loadtxt` lee valores numéricos; `skiprows=1` omite el encabezado y `ndmin=2`
mantiene al menos dos dimensiones, incluso para un archivo de una sola fila.
El orden de los nombres se conserva por acuerdo con el archivo, no dentro del
arreglo. La [ficha de datos](../datos/README.md) describe las unidades y su origen.

[Referencia de loadtxt](https://numpy.org/doc/stable/reference/generated/numpy.loadtxt.html).

In [3]:
room_names = ("north_room", "south_room", "storage_room")
readings = np.loadtxt(data_dir / "readings.csv", delimiter=",", skiprows=1, ndmin=2)
print(readings)

[[22. 24. 20.]
 [23. 25. 21.]
 [24. 26. 22.]
 [25. 27. 23.]
 [26. 28. 24.]
 [27. 29. 25.]
 [28. 30. 26.]
 [29. 31. 27.]]


**Observaciones sobre `readings`**

- **Dos corchetes `[[`:** el arreglo tiene dos dimensiones. El corchete exterior agrupa todas las filas y cada corchete interior es una fila (una ronda con las tres salas). El arreglo `[44. 48. 40.]` de la celda anterior tenía un solo corchete porque era de una dimensión: una sola fila de valores.
- **Valores con punto (`22.`):** aunque el archivo contiene enteros, `np.loadtxt` crea por defecto un arreglo de tipo decimal (`float64`). Ese tipo puede representar cualquier valor numérico, incluidos decimales y valores especiales como `nan` (dato faltante), por eso todos los números se muestran con punto.
- **Forma:** 8 filas (rondas) × 3 columnas (salas) = 24 temperaturas en total.

`ndim` cuenta ejes; `shape` indica cuántas posiciones hay en cada eje; `size`
es el total de elementos. `nbytes` cuenta los bytes del búfer de datos, no todos
los objetos ni el consumo total de Python.

```text
                        axis 1: salas →
                      norte   sur   almacén
axis 0: rondas ↓    0   22     24      20
                   1   23     25      21
                   2   24     26      22
                   …
```

[Propiedades de ndarray](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.html).

# Predicción: Dimensions = 3
# Predicción: Shape = (8, 3 )

In [4]:
print(f"Dimensions: {readings.ndim}")
print(f"Shape: {readings.shape}")
print(f"Elements: {readings.size}")
print(f"Data type: {readings.dtype}")
print(f"Bytes per element: {readings.itemsize}")
print(f"Data bytes: {readings.nbytes}")
assert readings.size == 8 * 3

Dimensions: 2
Shape: (8, 3)
Elements: 24
Data type: float64
Bytes per element: 8
Data bytes: 192


# Predicción scalar: 0 dimensiones, forma (), 1 elemento
#   Un solo número sin corchetes: no tiene ejes, por eso la forma es una tupla vacía.
# Predicción vector: 1 dimensión, forma (3,), 3 elementos
#   Un corchete = un eje con 3 posiciones. La coma en (3,) indica que es una tupla de un solo valor.
# Predicción matrix: 2 dimensiones, forma (1, 3), 3 elementos
#   Dos corchetes = dos ejes: 1 fila con 3 columnas.

In [5]:
scalar = np.array(22.0)
vector = np.array([22.0, 24.0, 20.0])
matrix = np.array([[22.0, 24.0, 20.0]])
for name, values in [("scalar", scalar), ("vector", vector), ("matrix", matrix)]:
    print(name, values.ndim, values.shape, values.size)

scalar 0 () 1
vector 1 (3,) 3
matrix 2 (1, 3) 3


### Ejercicio 1: una ronda, dos representaciones

Construye `one_round` como vector de tres temperaturas y `one_round_table`
como matriz de una fila. Comprueba las formas `(3,)` y `(1, 3)`. Explica por qué
ambos tienen tres elementos y distinto número de ejes.

Anticipa el resultado antes de ejecutar. Anota después qué comprobaste.

los valores en one_round son 3 datos en una sola línea. Por su parte, one_round_table son 3 datos asignados relacionados con la columna(que corresponde a la ubicación ya sea norte, sur, almacén) que sería el otro eje que permite una mejor organización, sobre todo si se van colectando más datos

In [6]:
# Write your solution here.

In [7]:
print(readings)

one_round = np.array([22.0, 24.0, 20.0])
one_round_table = np.array([[22.0, 24.0, 20.0]])
for name, values in [("one_round", one_round), ("one_round_table", one_round_table)]:
    print(name, values.ndim, values.shape, values.size)

assert one_round.shape == (3,)
assert one_round_table.shape == (1, 3)

[[22. 24. 20.]
 [23. 25. 21.]
 [24. 26. 22.]
 [25. 27. 23.]
 [26. 28. 24.]
 [27. 29. 25.]
 [28. 30. 26.]
 [29. 31. 27.]]
one_round 1 (3,) 3
one_round_table 2 (1, 3) 3


CONCLUSION:Los valores en `one_round` son 3 datos en una sola línea. Por su parte, `one_round_table` son 3 datos asignados a distintas columnas, que según el orden acordado en `room_names` corresponden a norte, sur y almacén, lo cual permite una mejor organización, sobre todo si se van colectando más datos.

Ambos tienen 3 elementos porque contienen los mismos valores: `size` cuenta cuántos datos hay. Lo que cambia es el número de ejes, que `ndim` cuenta como las direcciones de la estructura: el vector tiene 1 eje (una sola línea) y la matriz tiene 2 (filas y columnas), aunque por ahora solo tenga una fila.

## 3. Tipos numéricos

El tipo del arreglo determina cómo se representa cada elemento. `int64` guarda
enteros; `float64` permite decimales y valores especiales como `nan`.
Mezclar números con texto puede producir un arreglo de cadenas: crear el arreglo
no garantiza que pueda usarse en operaciones numéricas.

La asignación siguiente no convierte automáticamente el arreglo entero en uno
flotante. Observa qué sucede con la fracción. Para preservarla, crea primero un
arreglo flotante antes de asignar.

`np.testing.assert_allclose(actual, expected)` comprueba valores numéricos con
una pequeña tolerancia. `assert_array_equal` exige igualdad de valores y forma.
Si la comprobación pasa no imprime nada; si falla, lanza `AssertionError`.

[Tipos de datos de NumPy](https://numpy.org/doc/stable/user/basics.types.html).

In [8]:
integer_readings = np.array([22, 24, 20], dtype=np.int64)
integer_readings[0] = 22.75
print(integer_readings)

float_readings = np.array([22, 24, 20], dtype=np.float64)
float_readings[0] = 22.75
print(float_readings)
print(np.array([22, "missing"]).dtype)
np.testing.assert_allclose(float_readings[0], 22.75)
np.testing.assert_array_equal(integer_readings, [22, 24, 20])

[22 24 20]
[22.75 24.   20.  ]
<U21


### Ejercicio 2: preservar una temperatura decimal

Crea un arreglo con `22`, `24` y `20` que permita sustituir el primer valor por
`22.75` sin truncarlo. Comprueba el valor con `np.testing.assert_allclose`.

Anticipa el resultado antes de ejecutar. Anota después qué comprobaste.

In [9]:
# Write your solution here.

# Predicción: al crear el arreglo como decimal, el primer valor se reemplazará por 22.75 sin cortarse.

decimal_readings = np.array([22, 24, 20], dtype=float)
decimal_readings[0] = 22.75
print(decimal_readings)

np.testing.assert_allclose(decimal_readings[0], 22.75)

#Comprobación: el arreglo se creó con `dtype=float` (equivalente a `float64`), por lo que admite decimales y el valor `22.75` se conservó completo, como confirma `np.testing.assert_allclose`. En el ejemplo con `int64`, el mismo valor se truncó a `22` porque un arreglo de enteros no cambia su tipo al recibir un decimal: hay que definir el tipo decimal desde la creación.

[22.75 24.   20.  ]


## 4. Selección de filas y columnas

En `readings[row, column]`, el primer índice elige la ronda y el segundo la sala.
El slice excluye su límite superior. Un índice entero elimina ese eje del
resultado; un slice lo conserva.

[Selección de elementos y slicing](https://numpy.org/doc/stable/user/basics.indexing.html).

In [10]:
print(readings[0, 1])
print(readings[-1])
print(readings[:3, :2])
print(readings[:, 0])
print(readings[:, 0:1])
print(readings[:, 0].shape, readings[:, 0:1].shape)

24.0
[29. 31. 27.]
[[22. 24.]
 [23. 25.]
 [24. 26.]]
[22. 23. 24. 25. 26. 27. 28. 29.]
[[22.]
 [23.]
 [24.]
 [25.]
 [26.]
 [27.]
 [28.]
 [29.]]
(8,) (8, 1)


Una lista de índices permite escoger columnas no consecutivas. Comprueba el
orden resultante: `[2, 0]` coloca primero el almacén y después la sala norte.

In [11]:
selected_rooms = readings[:, [2, 0]]
print(selected_rooms[:2])
print(selected_rooms.shape)

[[20. 22.]
 [21. 23.]]
(8, 2)


### Ejercicio 3: últimas rondas

Obtén las últimas tres rondas para sala norte y almacén, en ese orden. El
resultado debe ser `[[27, 25], [28, 26], [29, 27]]`, de forma `(3, 2)`.
Comprueba valores y forma.

Anticipa el resultado antes de ejecutar. Anota después qué comprobaste.

In [12]:
# Write your solution here.

# Predicción: va a imprimir las rondas 6, 7 y 8 de sala norte y almacén, con forma (3, 2).

last_rounds = readings[-3:, [0, 2]]
print(last_rounds)
print(last_rounds.shape)

np.testing.assert_array_equal(last_rounds, [[27, 25], [28, 26], [29, 27]])
assert last_rounds.shape == (3, 2)
# Comprobé: los valores coinciden con los esperados y la forma es (3, 2).

[[27. 25.]
 [28. 26.]
 [29. 27.]]
(3, 2)


### Ejercicio 4: una columna con dos ejes

Selecciona la sala sur conservando la forma `(8, 1)`. Contrasta la expresión
con una selección que devuelva `(8,)`. Explica qué eje desapareció.

Anticipa el resultado antes de ejecutar. Anota después qué comprobaste.

In [13]:
# Write your solution here.

# Predicción: con 1:2 se conserva la forma (8, 1); con 1 solo queda (8,).

south_2d = readings[:, 1:2]
south_1d = readings[:, 1]
print(south_2d.shape, south_1d.shape)

assert south_2d.shape == (8, 1)
assert south_1d.shape == (8,)
# Explicación: desapareció el eje de las columnas (salas). Con el número entero 1
# se elige una sola sala y ese eje se elimina; con el rango 1:2 se conserva,
# aunque solo contenga una columna.

(8, 1) (8,)


## 5. Máscaras booleanas

Una comparación produce un booleano por elemento. Para seleccionar rondas usando
una sala, la máscara debe tener una posición por fila. `&` combina condiciones
por elemento; cada comparación necesita paréntesis.

In [14]:
north_temperatures = readings[:, 0]
warm_rounds = north_temperatures >= 27
print(warm_rounds)
print(warm_rounds.shape)
print(readings[warm_rounds])

comfortable_rounds = (north_temperatures >= 24) & (north_temperatures < 27)
print(readings[comfortable_rounds])

[False False False False False  True  True  True]
(8,)
[[27. 29. 25.]
 [28. 30. 26.]
 [29. 31. 27.]]
[[24. 26. 22.]
 [25. 27. 23.]
 [26. 28. 24.]]


Una máscara del mismo tamaño que toda la matriz selecciona celdas y devuelve
un vector. Ya no conserva una fila por ronda. Compara ambas selecciones antes de
usar el resultado en un resumen.

In [15]:
cell_mask = readings >= 28
print(cell_mask.shape)
print(readings[cell_mask])
print(readings[cell_mask].shape)
print(readings[readings[:, 1] >= 28].shape)

(8, 3)
[28. 29. 28. 30. 29. 31.]
(6,)
(4, 3)


### Ejercicio 5: un intervalo

Selecciona las rondas cuya temperatura de la sala sur esté entre `26` incluido
y `29` excluido. Deben quedar tres filas completas. Escribe la máscara, imprime
su forma y comprueba las temperaturas `[26, 27, 28]` de esa columna.

Anticipa el resultado antes de ejecutar. Anota después qué comprobaste.

In [16]:
# Write your solution here.

# Predicción: quedan 3 rondas completas (forma (3, 3)), con la sala sur en 26, 27 y 28.

south_temperatures = readings[:, 1]
interval_mask = (south_temperatures >= 26) & (south_temperatures < 29)
print(interval_mask.shape)

interval_rounds = readings[interval_mask]
print(interval_rounds)
print(interval_rounds.shape)

np.testing.assert_array_equal(interval_rounds[:, 1], [26, 27, 28])
assert interval_rounds.shape == (3, 3)
# Comprobé: la máscara tiene forma (8,), una posición por ronda, y se conservan
# 3 filas completas con las tres salas.

(8,)
[[24. 26. 22.]
 [25. 27. 23.]
 [26. 28. 24.]]
(3, 3)


## 6. Vistas y copias

Un slice básico produce una vista del mismo búfer. Modificar esa vista puede
cambiar el arreglo de origen. `.copy()` crea datos independientes; la selección
con una lista o un arreglo de índices enteros, o con una máscara booleana,
también obtiene una copia. Un índice entero aislado, como `readings[0]`,
puede devolver una vista de una fila.

Trabajaremos sobre `working_readings` para que el experimento no cambie
`readings`. Observa la diferencia entre alias de datos y archivos: ninguna de
estas asignaciones escribe el CSV.

[Copias y vistas](https://numpy.org/doc/stable/user/basics.copies.html).

In [17]:
working_readings = readings.copy()
north_view = working_readings[:, 0]
north_copy = working_readings[:, 0].copy()
print(np.shares_memory(working_readings, north_view))
print(np.shares_memory(working_readings, north_copy))
north_view[0] = -10
print(working_readings[0, 0], north_copy[0], readings[0, 0])

True
False
-10.0 22.0 22.0


In [18]:
warm_copy = readings[readings[:, 0] >= 27]
warm_copy[0, 0] = -20
print(np.shares_memory(readings, warm_copy))
print(readings[5, 0])

edited_readings = readings.copy()
edited_readings[edited_readings[:, 0] >= 27, 0] = 27
print(edited_readings[:, 0])

False
27.0
[22. 23. 24. 25. 26. 27. 27. 27.]


Obtener una selección con máscara crea una copia, pero asignar directamente
con `array[mask] = value` modifica el arreglo destino. La distinción está en qué
objeto recibe la asignación.

### Ejercicio 6: una selección independiente

Crea `last_rounds` con las dos últimas rondas, cambia su primera temperatura
a `100` y comprueba que `readings` sigue igual que antes. Usa `copy()` y
`np.testing.assert_array_equal` para demostrarlo.

Anticipa el resultado antes de ejecutar. Anota después qué comprobaste.

In [19]:
# Write your solution here.

# Predicción: al cambiar la copia a 100, readings no cambia.

readings_before = readings.copy()

last_rounds = readings[-2:].copy()
last_rounds[0, 0] = 100
print(last_rounds)

np.testing.assert_array_equal(readings, readings_before)
print(np.shares_memory(readings, last_rounds))
# Comprobé: last_rounds tiene 100 en su primera temperatura, readings sigue
# igual y no comparten memoria (False).

[[100.  30.  26.]
 [ 29.  31.  27.]]
False


## 7. Operaciones vectorizadas y resúmenes por eje

En una operación vectorizada expresamos qué cálculo aplicar al arreglo completo.
La conversión de Celsius a Fahrenheit conserva las ocho rondas y las tres salas.
El ciclo siguiente produce el mismo resultado y sirve para comparar las dos formas
de expresar el cálculo; no hace falta medir velocidad con solo 24 valores.

[Operaciones con arreglos](https://numpy.org/doc/stable/user/quickstart.html).


In [20]:
fahrenheit = readings * 9 / 5 + 32
fahrenheit_with_loop = np.array(
    [[temperature * 9 / 5 + 32 for temperature in row] for row in readings]
)
np.testing.assert_allclose(fahrenheit, fahrenheit_with_loop)
print(fahrenheit[:2])
print(fahrenheit.shape)

[[71.6 75.2 68. ]
 [73.4 77.  69.8]]
(8, 3)


### Ejercicio 7: convertir y conservar la forma

Crea `converted_readings` con todas las temperaturas en Fahrenheit mediante una
operación sobre el arreglo, sin escribir un ciclo. Comprueba la forma `(8, 3)`,
que la primera fila sea `[71.6, 75.2, 68.0]` y que `readings` no cambió.


In [21]:
# Write your solution here.

# Predicción: forma (8, 3) y primera fila [71.6, 75.2, 68.0].

readings_before = readings.copy()

converted_readings = readings * 9 / 5 + 32
print(converted_readings[:2])
print(converted_readings.shape)

assert converted_readings.shape == (8, 3)
np.testing.assert_allclose(converted_readings[0], [71.6, 75.2, 68.0])
np.testing.assert_array_equal(readings, readings_before)
# Comprobé: la operación se aplicó a las 24 temperaturas sin ciclo, la forma se
# conservó y readings no cambió.

[[71.6 75.2 68. ]
 [73.4 77.  69.8]]
(8, 3)


### Resumir por eje

Una reducción elimina el eje sobre el que opera. `axis=0` combina las rondas y
conserva las tres salas; `axis=1` combina las salas y conserva las ocho rondas.
Aquí todas las columnas tienen la misma unidad, así que una media por ronda tiene
sentido. Más adelante no promediaremos sin más columnas como edad y tarifa.

[Promedios con `numpy.mean`](https://numpy.org/doc/stable/reference/generated/numpy.mean.html).


In [22]:
room_means = readings.mean(axis=0)
round_means = readings.mean(axis=1)
print(room_means, room_means.shape)
print(round_means[:3], round_means.shape)
np.testing.assert_allclose(room_means, [25.5, 27.5, 23.5])

[25.5 27.5 23.5] (3,)
[22. 23. 24.] (8,)


### Broadcasting: una media distinta para cada sala

`readings` tiene forma `(8, 3)` y `room_means` forma `(3,)`. NumPy alinea las
tres posiciones con las tres columnas y resta cada media a las ocho rondas.
`room_means.reshape(1, 3)` muestra explícitamente esa fila de tres valores;
produce el mismo resultado. NumPy compara las dimensiones desde la derecha: en cada posición deben ser
iguales o una de ellas debe valer 1. Una forma `(8,)` no se alinearía con las
tres columnas. El ejemplo captura ese error para poder continuar.

[Reglas de broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html).


In [23]:
deviations = readings - room_means
explicit_row = room_means.reshape(1, 3)
print(readings.shape, room_means.shape, explicit_row.shape)
print(deviations[:2])
assert deviations.shape == readings.shape
np.testing.assert_allclose(deviations, readings - explicit_row)
np.testing.assert_allclose(deviations.mean(axis=0), [0, 0, 0], atol=1e-12)

try:
    readings - round_means
except ValueError as error:
    print("Incompatible shapes:", readings.shape, round_means.shape)
    print(error)

(8, 3) (3,) (1, 3)
[[-3.5 -3.5 -3.5]
 [-2.5 -2.5 -2.5]]
Incompatible shapes: (8, 3) (8,)
operands could not be broadcast together with shapes (8,3) (8,) 


## 8. Mediciones incompletas

`isfinite` identifica valores distintos de `nan` y los infinitos. `all(axis=1)`
exige que las tres posiciones de una ronda sean finitas. Usaremos esa política
para conservar las mismas rondas en todas las salas.

Es una regla del ejercicio: eliminar una fila también descarta mediciones válidas
de sus otras salas. La decisión debe quedar visible en el conteo de rechazos.

[isfinite](https://numpy.org/doc/stable/reference/generated/numpy.isfinite.html) ·
[all](https://numpy.org/doc/stable/reference/generated/numpy.all.html).

In [24]:
quality_readings = np.loadtxt(
    data_dir / "readings_quality.csv", delimiter=",", skiprows=1, ndmin=2
)
finite_cells = np.isfinite(quality_readings)
complete_rows = finite_cells.all(axis=1)
print(finite_cells)
print(complete_rows)
print(quality_readings[complete_rows])

[[ True  True  True]
 [ True False  True]
 [ True  True  True]
 [ True  True False]]
[ True False  True False]
[[22. 24. 20.]
 [24. 26. 22.]]


### Ejercicio 8: contar los rechazos

Cuenta las filas aceptadas y rechazadas de `quality_readings`. Comprueba que
son dos y dos, y que las medias del arreglo seleccionado son `[23, 25, 21]`.
¿Qué pasaría si no quedara ninguna fila completa?

Anticipa el resultado antes de ejecutar. Anota después qué comprobaste.

In [25]:
# Write your solution here.

# Predicción: 2 filas aceptadas y 2 rechazadas; medias [23, 25, 21].

accepted = int(complete_rows.sum())
rejected = int((~complete_rows).sum())
print(f"Accepted: {accepted}, rejected: {rejected}")

accepted_readings = quality_readings[complete_rows]
accepted_means = accepted_readings.mean(axis=0)
print(accepted_means)

assert accepted == 2
assert rejected == 2
np.testing.assert_allclose(accepted_means, [23, 25, 21])
# Comprobé: se rechazaron la ronda con nan y la ronda con inf.
# Si no quedara ninguna fila completa, el arreglo seleccionado tendría forma
# (0, 3) y la media daría [nan, nan, nan] con una advertencia de
# "Mean of empty slice". Habría que revisar el conteo antes de calcular medias.

Accepted: 2, rejected: 2
[23. 25. 21.]


In [26]:
# Ejercicio 9: diferencias respecto de la media
# Predicción: la forma será (8, 3) y la media de cada columna de diferencias será casi cero.

room_means_ex9 = readings.mean(axis=0)
differences = readings - room_means_ex9
print(room_means_ex9, room_means_ex9.shape)
print(differences[:3])
print(differences.shape)

assert differences.shape == (8, 3)
np.testing.assert_allclose(differences.mean(axis=0), [0, 0, 0], atol=1e-12)

# Explicación: readings tiene forma (8, 3) y las medias por sala (3,).
# NumPy compara las formas desde la derecha: el 3 coincide con las 3 salas,
# así que cada media se resta a su propia columna en las 8 rondas, sin ciclo.
# No se resta la media general (25.5) porque mezcla las tres salas: cada sala
# tiene su propio nivel de temperatura y queremos ver cuánto se aleja cada
# ronda del promedio de SU sala, no del promedio de todo el edificio.

[25.5 27.5 23.5] (3,)
[[-3.5 -3.5 -3.5]
 [-2.5 -2.5 -2.5]
 [-1.5 -1.5 -1.5]]
(8, 3)


## Continuación: llevar las operaciones al proyecto

En el proyecto `02-reporte-mediciones`, estas operaciones forman funciones que
se importan desde una notebook y desde `main.py`. Sigue la [guía](../GUIA.md) para
cambiar de proyecto y abrir `u2_n2_reporte_mediciones.ipynb`.

Antes de continuar, reinicia el kernel y ejecuta todas las celdas. Los resultados
no deben depender de haber ejecutado una celda fuera de orden.